In [1]:
import random

In [2]:
RANKS = "23456789TJQKA"
SUITS = "♠♥♣♦"

def newDeck():
    deck = [f"{rank}{suit}" for rank in RANKS for suit in SUITS]
    random.shuffle(deck)
    return deck


class Player:
    def __init__(self, name, chips):
        self.name = name
        self.chips = chips
        self.hand = []
        self.bet = 0

    def resetHand(self):
        self.hand = []
        self.bet = 0


class PokerGame:
    def __init__(self, players):
        self.players = [Player(name, chips) for name, chips in players]
        self.smallBlind = 1
        self.bigBlind = 2
        self.deck = []
        self.communityCards = []
        self.pot = 0
        self.bet = 0
        self.currentPlayerPosition = 0
        self.gameState = "null"
        self.preflopRaised = False

    def dealHoleCards(self):
        for _ in range(2):
            for player in self.players:
                player.hand.append(self.deck.pop(0))

    def dealFlop(self):
        self.deck.pop(0) # Burn card
        for _ in range(3):
            self.communityCards.append(self.deck.pop(0))
        self.currentPlayerPosition = 0
        for player in self.players:
            player.bet = 0

    def dealTurn(self):
        self.deck.pop(0) # Burn card
        self.communityCards.append(self.deck.pop(0))
        self.currentPlayerPosition = 0
        for player in self.players:
            player.bet = 0

    def dealRiver(self):
        self.dealTurn()

    def printSeparator(self, title):
        line = "=" * 70
        print(line)
        if title:
            print(f"{title}")
            print(line)

    def printPlayerStatus(self):
        print("PLAYER STATUS")
        print("Name                 | Chips    | Bet      | Hand")
        print("---------------------+----------+----------+----------------")
        for player in self.players:
            print(f"{player.name:<20} | ${player.chips:<7} | ${player.bet:<7} | {player.hand}")
        print("---------------------+----------+----------+----------------")

    def printRoundStatus(self):
        print(f"Pot: ${self.pot} | Current highest bet: ${self.bet}")
        print(f"Community cards: {self.communityCards}\n")

    def postBlinds(self):
        player = self.currentPlayer()
        self.players[0].chips -= self.smallBlind
        self.players[0].bet += self.smallBlind
        self.pot += self.smallBlind
        self.printSeparator("Posting blinds")
        print(f"Player {player.name} posted small blind of {self.smallBlind}.")
        self.advancePlayer()
        player = self.currentPlayer()
        self.players[1].chips -= self.bigBlind
        self.players[1].bet += self.bigBlind
        self.pot += self.bigBlind
        print(f"Player {player.name} posted big blind of {self.bigBlind}.")

        self.advancePlayer()

        self.printPlayerStatus()
        self.printRoundStatus()

    def currentPlayer(self):
        return self.players[self.currentPlayerPosition]

    def advancePlayer(self):
        self.currentPlayerPosition = (self.currentPlayerPosition + 1) % len(self.players)

    def action(self):
        player = self.currentPlayer()
        self.printPlayerStatus()
        self.printSeparator(f"{player.name}'s turn")
        self.printRoundStatus()
        print(f"Player hand: {player.hand}")
        bet = int(input(f"{player.name}, enter your bet: "))
        player.chips -= bet
        player.bet += bet
        self.pot += bet
        if (self.gameState == "pre-flop") and (bet > self.bigBlind):
            self.preflopRaised = True
        self.advancePlayer()
        self.printSeparator("Bet accepted")
        self.printPlayerStatus()

    def continueActionsUntilEqualBets(self):
        betArray = [player.bet for player in self.players]
        while len(set(betArray)) != 1:
            print("Players have not bet the same amount. Waiting for actions.")
            self.action()
            betArray = [player.bet for player in self.players]
        print("All players have matched the bet.")

    def startHand(self):
        self.deck = newDeck()
        self.communityCards = []
        self.pot = 0
        self.bet = 0
        self.currentPlayerPosition = 0
        self.gameState = "null"
        self.preflopRaised = False
        for player in self.players:
            player.resetHand()

        self.dealHoleCards()
        self.postBlinds()
        self.gameState = "pre-flop"
        self.continueActionsUntilEqualBets()

        # Gives big blind their action if no one has raised pre-flop
        if not self.preflopRaised:
            print("\nBig blind has an action. Waiting for actions.")
            self.action()
            betArray = [player.bet for player in self.players]
            self.continueActionsUntilEqualBets()

        self.dealFlop()
        self.action()
        self.continueActionsUntilEqualBets()

        self.dealTurn()
        self.action()
        self.continueActionsUntilEqualBets()

        self.dealRiver()
        self.action()
        self.continueActionsUntilEqualBets()


if __name__ == "__main__":
    game = PokerGame([("Player1", 1000), ("Player2", 1000), ("Player3", 1000)])
    game.startHand()


Posting blinds
Player Player1 posted small blind of 1.
Player Player2 posted big blind of 2.
PLAYER STATUS
Name                 | Chips    | Bet      | Hand
---------------------+----------+----------+----------------
Player1              | $999     | $1       | ['3♦', 'K♦']
Player2              | $998     | $2       | ['5♦', '8♥']
Player3              | $1000    | $0       | ['2♣', 'A♣']
---------------------+----------+----------+----------------
Pot: $3 | Current highest bet: $0
Community cards: []

Players have not bet the same amount. Waiting for actions.
PLAYER STATUS
Name                 | Chips    | Bet      | Hand
---------------------+----------+----------+----------------
Player1              | $999     | $1       | ['3♦', 'K♦']
Player2              | $998     | $2       | ['5♦', '8♥']
Player3              | $1000    | $0       | ['2♣', 'A♣']
---------------------+----------+----------+----------------
Player3's turn
Pot: $3 | Current highest bet: $0
Community cards: []

Pla